# 🤖 Model Building

This notebook trains baseline and advanced regression models for taxi trip duration prediction.

Models:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- XGBoost Regressor

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/train_featured.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1447928, 20)


,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day,pickup_month,pickup_weekday,is_weekend,is_rush_hour,is_night,haversine_distance_km,manhattan_distance_km,longitude_difference,latitude_difference,bearing
0,2,1,-73.982155,40.767937,-73.964630,40.765602,0,455,17,14,3,0,0,1,0,1.498521,1.735433,0.017525,-0.002335,99.970196
1,1,1,-73.980415,40.738564,-73.999481,40.731152,0,663,0,12,6,6,1,0,1,1.805507,2.430506,-0.019066,-0.007412,242.846232
2,2,1,-73.979027,40.763939,-74.005333,40.710087,0,2124,11,19,1,1,0,0,0,6.385098,8.203575,-0.026306,-0.053852,200.319835
3,2,1,-74.010040,40.719971,-74.012268,40.706718,0,429,19,6,4,2,0,1,0,1.485498,1.661331,-0.002228,-0.013252,187.262300
4,2,1,-73.973053,40.793209,-73.972923,40.782520,0,435,13,26,3,5,1,0,0,1.188588,1.199457,0.000130,-0.010689,179.473585


In [3]:
X = df.drop("trip_duration", axis=1)
y = df["trip_duration"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (1447928, 19)
Target Shape: (1447928,)


In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (1158342, 19)
X_valid: (289586, 19)


In [5]:
def evaluate_model(model, X_valid, y_valid):
    predictions = model.predict(X_valid)
    
    mae = mean_absolute_error(y_valid, predictions)
    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    r2 = r2_score(y_valid, predictions)
    
    return mae, rmse, r2

In [6]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_mae, linear_rmse, linear_r2 = evaluate_model(
    linear_model,
    X_valid,
    y_valid
)

print("Linear Regression Results")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R2 Score:", linear_r2)

Linear Regression Results
MAE: 273.3726359427516
RMSE: 419.7082339436068
R2 Score: 0.6029409019925507


In [7]:
decision_tree_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=15
)

decision_tree_model.fit(X_train, y_train)

dt_mae, dt_rmse, dt_r2 = evaluate_model(
    decision_tree_model,
    X_valid,
    y_valid
)

print("Decision Tree Results")
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)
print("R2 Score:", dt_r2)

Decision Tree Results
MAE: 207.43329985453315
RMSE: 360.6045301806694
R2 Score: 0.7068954706995383


In [8]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

rf_mae, rf_rmse, rf_r2 = evaluate_model(
    random_forest_model,
    X_valid,
    y_valid
)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2 Score:", rf_r2)

Random Forest Results
MAE: 184.71377822126541
RMSE: 317.62198953470534
R2 Score: 0.7726047982511619


In [9]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_mae, xgb_rmse, xgb_r2 = evaluate_model(
    xgb_model,
    X_valid,
    y_valid
)

print("XGBoost Results")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R2 Score:", xgb_r2)

XGBoost Results
MAE: 177.60360717773438
RMSE: 307.4584829696523
R2 Score: 0.7869247198104858


In [10]:
model_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        linear_mae,
        dt_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE": [
        linear_rmse,
        dt_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R2 Score": [
        linear_r2,
        dt_r2,
        rf_r2,
        xgb_r2
    ]
})

model_results.sort_values(by="RMSE")

,Model,MAE,RMSE,R2 Score
3,XGBoost,177.603607,307.458483,0.786925
2,Random Forest,184.713778,317.621990,0.772605
1,Decision Tree,207.433300,360.604530,0.706895
0,Linear Regression,273.372636,419.708234,0.602941


In [11]:
# Save all trained models
joblib.dump(linear_model, "../models/linear_regression.pkl")
joblib.dump(decision_tree_model, "../models/decision_tree.pkl")
joblib.dump(random_forest_model, "../models/random_forest.pkl")
joblib.dump(xgb_model, "../models/xgboost.pkl")

# Select best model based on lowest RMSE
best_model_name = model_results.sort_values(by="RMSE").iloc[0]["Model"]

if best_model_name == "Linear Regression":
    best_model = linear_model
elif best_model_name == "Decision Tree":
    best_model = decision_tree_model
elif best_model_name == "Random Forest":
    best_model = random_forest_model
else:
    best_model = xgb_model

joblib.dump(best_model, "../models/best_model.pkl")

print(f"Best model selected: {best_model_name}")
print("All models and best model saved successfully!")

Best model selected: XGBoost
All models and best model saved successfully!


# Conclusion

Model building is complete.

Four regression models were trained and compared:

- Linear Regression
- Decision Tree
- Random Forest
- XGBoost

All trained models were saved in the `models/` directory.

The best-performing model was selected based on the lowest RMSE and saved as:

`models/best_model.pkl`

Next step: Model Evaluation and Hyperparameter Tuning.